In [ ]:
"""
TRAIN NOTEBOOK - PHUONG PHAP 2 (RNN_HYBRID).
Su dung mo hinh moi trong MMDF_/models_v2/RNN_HYBRID.py.
KHONG sua file train.ipynb goc. Dung chung dataset/optimizer/scheduler/loss weights.
"""
from ruamel.yaml import YAML
import numpy as np
import os

import torch
import torch.backends.cudnn as cudnn

from transformers import BertTokenizerFast
from MMDF_ import utils
from MMDF_.dataset import create_dataset, create_loader
from MMDF_.scheduler import create_scheduler
from MMDF_.optim import create_optimizer

from tqdm import tqdm
from collections import OrderedDict
from accelerate import Accelerator

# === Mo hinh moi (phuong phap 2) ===
from MMDF_.models_v2 import RNN_HYBRID
from MMDF_.utils import text_input_adjust, AttrDict


In [ ]:
args = {
    'config': r'MMDF_\configs\train_v2.yaml',
    'checkpoint': '',
    'resume': False,
    'output_dir': 'results_v2',
    'text_encoder': 'bert-base-uncased',
    'device': 'cuda',
    'world_size': 1,
    'launcher': 'pytorch',
    'model_save_epoch': 1,
    'token_momentum': False,
}
args = AttrDict(args)
yaml = YAML(typ='safe')

with open(args.config, 'r') as f:
    config = yaml.load(f)


In [ ]:
def train(args, model, data_loader, optimizer, tokenizer, epoch, EPOCH, warmup_steps, device, scheduler, config, accelerator):
    model.train()

    step_size = 100
    warmup_iterations = warmup_steps * step_size
    avg_loss = []

    pbar = tqdm(data_loader)
    for i, (image, label, text, fake_image_box, fake_word_pos, W, H) in enumerate(pbar):
        pbar.set_description(f"[v2] Epoch {epoch}/{EPOCH}")

        if config['schedular']['sched'] == 'cosine_in_step':
            scheduler.adjust_learning_rate(optimizer, i / len(data_loader) + epoch, args, config)

        optimizer.zero_grad()

        image = image.to(device, non_blocking=True)
        fake_image_box = fake_image_box.to(device)

        text_input = tokenizer(text, max_length=128, truncation=True, add_special_tokens=True,
                               return_attention_mask=True, return_token_type_ids=False)
        text_input, fake_token_pos = text_input_adjust(text_input, fake_word_pos, device)

        if epoch > 0:
            alpha = config['alpha']
        else:
            alpha = config['alpha'] * min(1, i / len(data_loader))

        loss_MAC, loss_BIC, loss_bbox, loss_giou, loss_TMG, loss_MLC = model(
            image, label, text_input, fake_image_box, fake_token_pos, alpha=alpha
        )

        loss = (config['loss_MAC_wgt'] * loss_MAC
                + config['loss_BIC_wgt'] * loss_BIC
                + config['loss_bbox_wgt'] * loss_bbox
                + config['loss_giou_wgt'] * loss_giou
                + config['loss_TMG_wgt'] * loss_TMG
                + config['loss_MLC_wgt'] * loss_MLC)

        avg_loss.append(loss.item())
        accelerator.backward(loss)
        optimizer.step()

        pbar.set_postfix(OrderedDict([
            ('MAC', f'{loss_MAC.item():.3f}'),
            ('BIC', f'{loss_BIC.item():.3f}'),
            ('bbox', f'{loss_bbox.item():.3f}'),
            ('giou', f'{loss_giou.item():.3f}'),
            ('TMG', f'{loss_TMG.item():.3f}'),
            ('MLC', f'{loss_MLC.item():.3f}'),
            ('loss', f'{loss.item():.3f}'),
            ('AVG', f'{np.mean(avg_loss):.3f}'),
            ('lr', f"{optimizer.param_groups[0]['lr']}"),
        ]))

        if epoch == 0 and i % step_size == 0 and i <= warmup_iterations and config['schedular']['sched'] != 'cosine_in_step':
            scheduler.step(i // step_size)


In [ ]:
accelerator = Accelerator()
device = accelerator.device
cudnn.benchmark = True

start_epoch = 0
max_epoch = config['schedular']['epochs']
warmup_steps = config['schedular']['warmup_epochs']

train_dataset, val_dataset = create_dataset(config)
train_loader, val_loader = create_loader(
    [train_dataset, val_dataset],
    batch_size=[config['batch_size_train']] + [config['batch_size_val']],
    num_workers=[4, 4],
    is_trains=[True, False],
)

tokenizer = BertTokenizerFast.from_pretrained(args.text_encoder)

model = RNN_HYBRID(args=args, config=config, text_encoder=args.text_encoder,
                   tokenizer=tokenizer, init_deit=True)
model = model.to(device)


In [ ]:
arg_opt = utils.AttrDict(config['optimizer'])
optimizer = create_optimizer(arg_opt, model)
arg_sche = utils.AttrDict(config['schedular'])
lr_scheduler, _ = create_scheduler(arg_sche, optimizer)
if config['schedular']['sched'] == 'cosine_in_step':
    args.lr = config['optimizer']['lr']


In [ ]:
model, optimizer, train_loader, val_loader, lr_scheduler = accelerator.prepare(
    model, optimizer, train_loader, val_loader, lr_scheduler
)


In [ ]:
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def quick_eval(model, loader, tokenizer, device):
    model.eval()
    y_true, y_pred, acc_n, acc_c = [], [], 0, 0
    for image, label, text, fake_image_box, fake_word_pos, W, H in tqdm(loader, desc='[v2] val'):
        image = image.to(device, non_blocking=True)
        text_input = tokenizer(text, max_length=128, truncation=True, add_special_tokens=True,
                               return_attention_mask=True, return_token_type_ids=False)
        text_input, fake_token_pos = text_input_adjust(text_input, fake_word_pos, device)
        logits_real_fake, _, _, _ = model(image, label, text_input, fake_image_box,
                                          fake_token_pos, is_train=False)
        cls_label = torch.ones(len(label), dtype=torch.long, device=device)
        real_pos = np.where(np.array(label) == 'orig')[0].tolist()
        cls_label[real_pos] = 0
        y_pred.extend(F.softmax(logits_real_fake, dim=1)[:, 1].cpu().tolist())
        y_true.extend(cls_label.cpu().tolist())
        acc_c += (logits_real_fake.argmax(1) == cls_label).sum().item()
        acc_n += cls_label.size(0)
    auc = roc_auc_score(y_true, y_pred) if len(set(y_true)) > 1 else float('nan')
    return {'AUC_cls': auc * 100, 'ACC_cls': acc_c / acc_n * 100}


In [ ]:
os.makedirs(args.output_dir, exist_ok=True)
os.makedirs('checkpoints_v2', exist_ok=True)

for epoch in range(start_epoch, 3):
    print(f"\n[v2] === train epoch {epoch+1} ===")
    train(args, model, train_loader, optimizer, tokenizer, epoch + 1, max_epoch,
          warmup_steps, device, lr_scheduler, config, accelerator)

    print(f"[v2] === val epoch {epoch+1} ===")
    val_stats = quick_eval(model, val_loader, tokenizer, device)
    print(val_stats)
    with open(os.path.join(args.output_dir, f"result_{epoch+1}.txt"), 'w') as f:
        f.write(str(val_stats))

    torch.save({'model': model.state_dict(), 'epoch': epoch + 1, 'config': config},
               os.path.join('checkpoints_v2', f'rnn_hybrid_{epoch+1:02d}.pth'))
